# SHAP and Business Feature Analysis

This notebook documents the XGBoost FE SHAP workflow plus the business bucketing analysis used for feature interpretation. It is intentionally kept as the single feature-analysis notebook for the repository.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "2_feature_engineering"))
from run_xgboost_fe import RNG_SEED, add_fe

TRAIN_PATH = ROOT / "train_clean.csv"
TEST_PATH = ROOT / "test_clean.csv"
OUT_DIR = ROOT / "7_feature_analysis"
OUT_DIR.mkdir(exist_ok=True)


## Load Feature-Engineered Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train_fe, test_fe = add_fe(train.copy(), test.copy())

target_col = "TARGET"
id_col = "ID"
feature_cols = [c for c in train_fe.columns if c not in [id_col, target_col]]
X = train_fe[feature_cols]
y = train_fe[target_col]

print(X.shape, y.mean())


## Train XGBoost FE Model for SHAP

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    random_state=RNG_SEED,
    n_jobs=-1,
)
model.fit(X, y)


## SHAP Summary and Dependence Plots

In [ ]:
import shap

sample = X.sample(n=min(5000, len(X)), random_state=RNG_SEED)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)

mean_abs = np.abs(shap_values).mean(axis=0)
shap_summary = (
    pd.DataFrame({"feature": sample.columns, "mean_abs_shap": mean_abs})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_summary["shap_pct"] = shap_summary["mean_abs_shap"] / shap_summary["mean_abs_shap"].sum()
shap_summary.head(20).to_csv(OUT_DIR / "xgboost_fe_top20_shap.csv", index=False)

shap.summary_plot(shap_values, sample, show=False, max_display=20)
plt.tight_layout()
plt.savefig(OUT_DIR / "xgboost_fe_top20_shap_summary.png", dpi=200, bbox_inches="tight")
plt.close()

for feature in shap_summary.head(6)["feature"]:
    shap.dependence_plot(feature, shap_values, sample, show=False)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"xgboost_fe_shap_dependence_{feature}.png", dpi=200, bbox_inches="tight")
    plt.close()

shap_summary.head(20)


## Business Bucketing Analysis

In [ ]:
overall_rate = train["TARGET"].mean()

def bucket_stats(df, bucket_col):
    out = (
        df.groupby(bucket_col, observed=False)["TARGET"]
        .agg(customers="count", dissatisfied_customers="sum", dissatisfied_rate="mean")
        .reset_index()
    )
    out["lift_vs_overall"] = out["dissatisfied_rate"] / overall_rate
    return out

train_biz = train.copy()
train_biz["var15_bucket"] = pd.cut(
    train_biz["var15"],
    bins=[0, 23, 30, 40, 50, 60, 120],
    labels=["under_23", "23_30", "30_40", "40_50", "50_60", "60_plus"],
    right=False,
)
train_biz["saldo_var30_binary"] = np.where(train_biz["saldo_var30"] <= 0, "zero_or_negative", "positive")
train_biz["ind_var30_binary"] = np.where(train_biz["ind_var30"] == 0, "no_core_product", "has_core_product")

business_tables = {
    "var15_age": bucket_stats(train_biz, "var15_bucket"),
    "saldo_var30_binary": bucket_stats(train_biz, "saldo_var30_binary"),
    "ind_var30": bucket_stats(train_biz, "ind_var30_binary"),
}

for name, table in business_tables.items():
    table.to_csv(OUT_DIR / f"business_{name}.csv", index=False)

business_tables["var15_age"]
